# 02 · 特征工程

**原则**：只给模型加入它**无法通过阈值切分自己得到**的信息；全部是确定性变换，不使用标签，不删除也不修改原始 56 列。数据检查（01）表明数据没有错误需要清洗，所以本项目的数据处理都是「加信息」。

共三轮：
1. **第一轮**：DTI 数值化、LTV 重算、收入比值、缺失组合、地区频次；
2. **第二轮**：沿第一轮最有效的 LTV 方向继续——LTV 申报精度、产品门槛、反推房价、期限；
3. **第三轮**：由残差诊断（05）提出的候选——产品标志组合、LTV 区间一致性、风险叠加、相对普查区收入。

每一轮都用**消融实验**评估：同一套快速 CatBoost 配置、固定五折，只比较同一折上相对基线的 AUC 差；五折平均提升达标且至少 4/5 折为正才保留。最后把全部特征块保存为**特征库**（`artifacts/02_feature_engineering/feature_store/`），供 03–05 使用。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 原始数据读取、审计、探索分析、数据质量检查、固定五折 |
| **02_feature_engineering** | 全部特征的定义、三轮特征消融、保存特征库 |
| **03_model_selection** | 基线模型（逻辑回归、XGBoost、CatBoost、AutoGluon）、线性模型的数值变换、CatBoost 调参与训练方式筛选 |
| **04_final_model** | 最终五折模型、多种子、学习曲线、全量重训与最终提交 |
| **05_error_analysis** | 残差诊断、LightGBM 与神经网络、跨实验融合、伪标签、公开榜噪声与运行间差异 |

原始数据在 `data/`；每本 notebook 的结果保存在 `artifacts/` 下与它同名的目录。

In [ ]:
# 环境、路径与通用函数（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def find_project_dir():
    candidates = []
    if os.environ.get('HMDA_PROJECT_DIR'):
        candidates.append(Path(os.environ['HMDA_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 HMDA_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                                        # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'                                  # 每本 notebook 的结果写入同名子目录
PREPARED_DIR = ARTIFACTS / '01_data_exploration'                       # 01：固定数据接口
FEATURE_DIR = ARTIFACTS / '02_feature_engineering' / 'feature_store'   # 02：特征库
SEED = 42
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def ensure_catboost():
    """只在训练时调用：安装并导入固定版本的 CatBoost。"""
    global catboost
    try:
        installed = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已导入另一版 CatBoost；请重启运行时后从头运行')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
        importlib.invalidate_caches()
    import catboost as catboost_module
    if catboost_module.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}，实际为 {catboost_module.__version__}')
    catboost = catboost_module
    return catboost_module


def ensure_gpu():
    """只在训练时调用：确认 Colab 分配了 NVIDIA GPU。"""
    global GPU_NAME
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('训练需要 GPU：Colab「更改运行时类型」选 T4 GPU 后从头运行')
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError('NVIDIA GPU 不可用：请选择 T4 GPU 后从头运行')
    GPU_NAME = probe.stdout.strip().splitlines()[0]
    return GPU_NAME


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图（Colab 普通 CPU 运行时即可，几分钟跑完）。把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应部分；每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。

In [ ]:
TRAIN = {
    'ablation_round1': False,   # 第一轮消融：7 个变体 × 5 折，约 70 分钟
    'ablation_round2': False,   # 第二轮消融：8 个变体 × 5 折，约 80 分钟
    'ablation_round3': False,   # 第三轮消融：6 个变体 × 5 折，约 60 分钟
}

## 1. 读取并核对固定数据接口

核对 01 生成的 `artifacts/01_data_exploration/`：SHA-256、列顺序、ID 和固定五折。

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_data_exploration.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = DATA_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    DATA_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    DATA_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

## 2. 第一轮特征

### DTI 上下界

保留原始 DTI 类别；开放区间和缺失的未知端为 `NaN`，遇到未见过的类别直接报错。

In [ ]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

### 第一轮特征块

比值的分母必须大于 0，否则为缺失，不加任意 epsilon。


| 特征块 | 新增列 | 想法 |
|---|---|---|
| `dti` | `dti_lower`、`dti_upper` | DTI 类别（如 `30%-<36%`）对应的数值上下界 |
| `ltv` | `ltv_calc`、`ltv_gap` | 由贷款额 / 房价**重新算出** LTV，以及它与申报 LTV 的差 |
| `income` | `income_to_area`、`loan_to_income`、`value_to_income` | 收入相对地区收入、贷款额和房价相对收入 |
| `missing` | `missing_pattern`（类别）、`missing_count` | 关键字段的缺失组合 |
| `freq` | `freq_county_code`、`freq_derived_msa-md`、`freq_state_code` | 地区在训练 + 测试中出现的次数（不使用标签） |

In [ ]:
def safe_ratio(numerator, denominator):
    valid = numerator.notna() & denominator.notna() & (denominator > 0)
    return (numerator / denominator.where(valid)).where(valid).astype('float64')


def build_blocks(train, test):
    both = pd.concat([train, test], ignore_index=True)
    blocks = {}
    blocks['dti'] = add_dti_bounds(both)[['dti_lower', 'dti_upper']]

    ltv_calc = 100 * safe_ratio(both['loan_amount'], both['property_value'])
    blocks['ltv'] = pd.DataFrame({
        'ltv_calc': ltv_calc,
        'ltv_gap': both['loan_to_value_ratio'] - ltv_calc,
    })

    income_dollars = both['income'] * 1000      # HMDA 收入单位为千美元
    blocks['income'] = pd.DataFrame({
        'income_to_area': safe_ratio(income_dollars, both['ffiec_msa_md_median_family_income']),
        'loan_to_income': safe_ratio(both['loan_amount'], income_dollars),
        'value_to_income': safe_ratio(both['property_value'], income_dollars),
    })

    missing = pd.DataFrame({
        'ltv': both['loan_to_value_ratio'].isna(),
        'pv': both['property_value'].isna(),
        'inc': both['income'].isna(),
        'term': both['loan_term'].isna(),
        'dti': both['debt_to_income_ratio'].eq('MISSING'),
        'age': both['applicant_age'].eq('MISSING'),
    })
    blocks['missing'] = pd.DataFrame({
        'missing_pattern': 'm' + missing.astype(int).astype(str).agg(''.join, axis=1),
        'missing_count': missing.sum(axis=1).astype('float64'),
    })

    freq = {}
    for column in ['county_code', 'derived_msa-md', 'state_code']:
        counts = both[column].map(both[column].value_counts()).astype('float64')
        freq[f'freq_{column}'] = counts.where(both[column].ne('MISSING'))
    blocks['freq'] = pd.DataFrame(freq)

    n_train = len(train)
    out = {}
    for name, frame in blocks.items():
        require(len(frame) == len(both), f'{name} 行数不符')
        out[name] = (frame.iloc[:n_train].reset_index(drop=True),
                     frame.iloc[n_train:].reset_index(drop=True))
    return out


BLOCKS = build_blocks(X_base, X_test_base)
NEW_CATEGORICAL = {'missing_pattern'}


def variant_frames(blocks):
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


for name, (train_block, _) in BLOCKS.items():
    described = train_block.describe(include='all').T
    print(f'\n[{name}] 缺失率：', train_block.isna().mean().round(3).to_dict())
    display(described.iloc[:, :6])

## 3. 第二轮特征

`ltv_precision` 需要申报 LTV 的**原始文本**（例如 `80.0` 与 `79.85`），所以按字符串重新读取这一列。


| 特征块 | 新增列 | 依据 |
|---|---|---|
| `precision` | `ltv_precision`（缺失 / 整数 / 1–3 位小数 / 更多） | LTV 申报为整数时批准率 90.7%，两位小数 84.4%：反映不同机构的申报习惯，树模型无法从数值本身切出「是否整数」 |
| `thresholds` | `ltv_band`（类别）、`ltv_over_program` | 80、95、97、100 是常见产品门槛；`ltv_over_program` = LTV − 该贷款类型通常上限（常规 97、FHA 96.5、VA/RHS 100） |
| `implied` | `implied_property_value`、`implied_value_ratio` | 由贷款额和申报 LTV 反推房价，再与申报房价比较 |
| `term` | `term_is_360`、`term_years` | 贷款期限是否为 30 年 |

In [ ]:
def read_raw_ltv(path):
    text = pd.read_csv(path, usecols=['loan_to_value_ratio'], dtype=str,
                       keep_default_na=False)['loan_to_value_ratio']
    return text


raw_ltv = pd.concat([read_raw_ltv(PREPARED_DIR / 'train_features.csv'),
                     read_raw_ltv(PREPARED_DIR / 'test_features.csv')], ignore_index=True)
both = pd.concat([X_base, X_test_base], ignore_index=True)
require(len(raw_ltv) == len(both), 'LTV 文本行数不符')
require(pd.to_numeric(raw_ltv.replace('', np.nan)).equals(both['loan_to_value_ratio'])
        or np.allclose(pd.to_numeric(raw_ltv.replace('', np.nan)), both['loan_to_value_ratio'],
                       equal_nan=True), 'LTV 文本与数值不一致')

decimals = raw_ltv.str.split('.').str[1].fillna('')
precision = np.select(
    [raw_ltv.eq(''), decimals.isin(['', '0']), decimals.str.len().eq(1),
     decimals.str.len().eq(2), decimals.str.len().eq(3)],
    ['missing', 'integer', 'dec1', 'dec2', 'dec3'], default='dec4plus',
)
ltv = both['loan_to_value_ratio']
band = np.select(
    [ltv.isna(), ltv < 80, ltv == 80, ltv < 95, ltv < 97, ltv < 100, ltv == 100],
    ['missing', 'lt80', 'eq80', '80_95', '95_97', '97_100', 'eq100'], default='gt100',
)
PROGRAM_MAX_LTV = {'1': 97.0, '2': 96.5, '3': 100.0, '4': 100.0}   # 常规/FHA/VA/RHS
unknown_types = set(both['loan_type']) - set(PROGRAM_MAX_LTV)
require(not unknown_types, f'未知 loan_type：{sorted(unknown_types)}')
implied_value = safe_ratio(both['loan_amount'] * 100, ltv)
term = both['loan_term']

round2 = {
    'precision': pd.DataFrame({'ltv_precision': precision}),
    'thresholds': pd.DataFrame({
        'ltv_band': band,
        'ltv_over_program': ltv - both['loan_type'].map(PROGRAM_MAX_LTV).astype('float64'),
    }),
    'implied': pd.DataFrame({
        'implied_property_value': implied_value,
        'implied_value_ratio': safe_ratio(implied_value, both['property_value']),
    }),
    'term': pd.DataFrame({
        'term_is_360': term.eq(360).astype('float64').where(term.notna()),
        'term_years': (term / 12).round(1),
    }),
}
n_train = len(X_base)
for name, frame in round2.items():
    require(len(frame) == len(both), f'{name} 行数不符')
    BLOCKS[name] = (frame.iloc[:n_train].reset_index(drop=True),
                    frame.iloc[n_train:].reset_index(drop=True))
NEW_CATEGORICAL |= {'ltv_precision', 'ltv_band'}

train_labels_check = y.to_numpy()
for column, frame in [('ltv_precision', round2['precision']), ('ltv_band', round2['thresholds'])]:
    table = pd.crosstab(frame[column].iloc[:n_train], train_labels_check, normalize='index')
    counts = frame[column].iloc[:n_train].value_counts()
    display(table.assign(rows=counts).round(3))
for name in ['implied', 'term']:
    display(BLOCKS[name][0].describe().T.round(3))
del both

## 4. 第三轮特征


| 特征块 | 新增列 | 依据 |
|---|---|---|
| `purpose` | `product_flags`（7 个产品标志拼成一个类别） | 残差诊断中离散度最高的分组（仅 16 组，证据弱） |
| `ltv_consistency` | `ltv_consistency`、`ltv_outside_interval` | 贷款额与房价都是 1 万美元区间的中点，由它们只能确定 LTV 的区间；申报值落在区间外说明房价口径不同 |
| `risk_layers` | `risk_layer_count`、`risk_layer_pattern` | 审批中常见的「风险叠加」：高 LTV、高 DTI、未用 AUS、非自住、LTV 缺失、次级留置 |
| `tract_income` | `income_to_tract` | 收入相对所在普查区的中位收入 |

In [ ]:
both = pd.concat([X_base, X_test_base], ignore_index=True)
ltv = both['loan_to_value_ratio']
loan, value = both['loan_amount'], both['property_value']

# HMDA 公开数据中贷款额与房价都是 1 万美元区间的中点（训练集全部以 5000 结尾），
# 所以由二者算出的 LTV 只能确定一个区间；申报 LTV 落在区间外，说明用的房价口径不同。
require(((loan % 10000) == 5000).all(), '贷款额不全是 1 万区间中点，区间假设不成立')
ltv_min = 100 * safe_ratio(loan - 5000, value + 5000)
ltv_max = 100 * safe_ratio(loan + 5000, value - 5000)
outside = np.where(ltv.isna() | ltv_min.isna(), np.nan,
                   np.where(ltv < ltv_min, ltv - ltv_min, np.where(ltv > ltv_max, ltv - ltv_max, 0.0)))
consistency = np.select([ltv.isna() | ltv_min.isna(), outside == 0, outside < 0],
                        ['unknown', 'inside', 'below'], default='above')

PRODUCT_FLAGS = ['business_or_commercial_purpose', 'open-end_line_of_credit', 'reverse_mortgage',
                 'interest_only_payment', 'balloon_payment', 'negative_amortization',
                 'other_nonamortizing_features']
purpose = both[PRODUCT_FLAGS].astype(str).agg('|'.join, axis=1)

dti_lower = add_dti_bounds(both)['dti_lower']
risk = pd.DataFrame({
    'high_ltv': ltv > 95,
    'high_dti': dti_lower >= 45,                 # 下界 ≥ 45 才确定为高 DTI
    'no_aus': both['aus-1'].eq('6'),             # aus-1 = 6：未使用自动审批系统
    'not_owner_occupied': both['occupancy_type'].ne('1'),
    'ltv_missing': ltv.isna(),
    'second_lien': both['lien_status'].eq('2'),
})
tract_median_income = both['ffiec_msa_md_median_family_income'] * both['tract_to_msa_income_percentage'] / 100

round3 = {
    'purpose': pd.DataFrame({'product_flags': 'p' + purpose}),
    'ltv_consistency': pd.DataFrame({'ltv_consistency': consistency,
                                     'ltv_outside_interval': outside}),
    'risk_layers': pd.DataFrame({'risk_layer_count': risk.sum(axis=1).astype('float64'),
                                 'risk_layer_pattern': 'r' + risk.astype(int).astype(str).agg(''.join, axis=1)}),
    'tract_income': pd.DataFrame({'income_to_tract': safe_ratio(both['income'] * 1000, tract_median_income)}),
}
n_train = len(X_base)
for name, frame in round3.items():
    require(len(frame) == len(both), f'{name} 行数不符')
    BLOCKS[name] = (frame.iloc[:n_train].reset_index(drop=True),
                    frame.iloc[n_train:].reset_index(drop=True))
NEW_CATEGORICAL |= {'product_flags', 'ltv_consistency', 'risk_layer_pattern'}

for column, block in [('product_flags', 'purpose'), ('ltv_consistency', 'ltv_consistency'),
                      ('risk_layer_count', 'risk_layers')]:
    values = BLOCKS[block][0][column]
    table = pd.crosstab(values, y.to_numpy(), normalize='index').assign(rows=values.value_counts())
    display(table.sort_values('rows', ascending=False).head(10).round(3))
display(BLOCKS['tract_income'][0].describe().T.round(3))
del both

## 5. 保存特征库

03–05 读取这里保存的特征，不再重复定义。文件带 SHA-256 与数据指纹，数据变化时会拒绝读取。

In [ ]:
# 保存特征库：全部特征块的新列（不含原始 56 列，原始列由 01 的固定数据接口提供）
FEATURE_SETS = {
    'ROUND1_ALL': ['dti', 'ltv', 'income', 'missing', 'freq'],                                   # 第一轮全部（68 列）
    'G7': ['dti', 'ltv', 'income', 'missing', 'precision', 'thresholds', 'implied', 'term'],      # 最终 M1/M2（72 列）
    'LEAN': ['dti', 'ltv', 'income', 'missing', 'precision', 'thresholds'],                      # 最终 M3（68 列）
}
block_columns = {name: list(train_block.columns) for name, (train_block, _) in BLOCKS.items()}
all_columns = [c for columns in block_columns.values() for c in columns]
require(len(all_columns) == len(set(all_columns)), '不同特征块之间有重名列')
train_store = pd.concat([BLOCKS[name][0] for name in block_columns], axis=1)
test_store = pd.concat([BLOCKS[name][1] for name in block_columns], axis=1)
require(train_store.shape == (len(X_base), len(all_columns))
        and test_store.shape == (len(X_test_base), len(all_columns)), '特征库形状不符')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
store_paths = {'train': FEATURE_DIR / 'blocks_train.parquet', 'test': FEATURE_DIR / 'blocks_test.parquet'}
train_store.to_parquet(store_paths['train'], index=False)
test_store.to_parquet(store_paths['test'], index=False)
write_json(FEATURE_DIR / 'feature_store.json', {
    'blocks': block_columns,
    'categorical': sorted(c for c in NEW_CATEGORICAL if c in all_columns),
    'feature_sets': FEATURE_SETS,
    'rows': {'train': len(train_store), 'test': len(test_store)},
    'data_hashes': DATA_HASHES,
    'files': {path.name: sha256_file(path) for path in store_paths.values()},
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
})
for part, frame in [('train', train_store), ('test', test_store)]:
    restored = pd.read_parquet(store_paths[part])
    pd.testing.assert_frame_equal(restored.reset_index(drop=True), frame.reset_index(drop=True), check_dtype=False)
display(pd.DataFrame([{'特征块': name, '列数': len(cols), '列': ', '.join(cols)} for name, cols in block_columns.items()]))
print(f'特征库已保存到 {FEATURE_DIR}：{len(all_columns)} 列，回读核对通过。')

## 6. 消融实验

**快速配置**：CatBoost GPU，depth 8、学习率 0.05、L2=3、Bernoulli 抽样 0.8、固定 1600 轮。每个变体 = 原始 56 列 + 若干特征块；五折上与基线**同折配对**比较。下表 `delta_vs_base` 为五折平均的 AUC 变化，`folds_better` 为 5 折中优于基线的折数，`keep` 为是否达到保留标准。

In [ ]:
# 只读的结果工具：读取每个（模型/变体, 折）保存的 JSON，做同折配对比较
def load_runs(directory, pattern='*__fold?.json'):
    directory = Path(directory)
    paths = sorted(directory.glob(pattern))
    require(paths, f'{directory} 中没有结果文件：请确认 artifacts 已从 Drive 同步，或打开对应的训练开关')
    return pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in paths])


def paired_table(runs, key, baseline, min_gain=0.0005, min_folds=4):
    auc = runs.pivot(index='fold', columns=key, values='valid_auc')
    require(baseline in auc.columns, f'缺少基线 {baseline}')
    delta = auc.sub(auc[baseline], axis=0)
    table = pd.DataFrame({
        'folds': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'delta_vs_base': delta.mean(),
        'folds_better': (delta > 0).sum(),
    })
    table['keep'] = ((table['folds'] == 5) & (table['delta_vs_base'] >= min_gain)
                     & (table['folds_better'] >= min_folds))
    table.loc[baseline, 'keep'] = False
    return table.sort_values('mean_auc', ascending=False), delta


def plot_deltas(delta, baseline, title):
    others = delta.drop(columns=[baseline])
    order = others.mean().sort_values().index.tolist()
    means = others[order].mean() * 1e4
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.6))
    ax.barh(range(len(order)), means, color=['#4c72b0' if v > 0 else '#c44e52' for v in means])
    for i, name in enumerate(order):
        ax.scatter(others[name] * 1e4, np.full(len(others), i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(range(len(order)), order)
    ax.set_xlabel(f'AUC change vs {baseline} (x1e-4); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

### 6.1 第一轮（基线 = 原始 56 列）

保留标准：平均 ≥ +0.0003 且 ≥ 4/5 折。

In [ ]:
EXPERIMENT_ID = 'ablation_round1'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None             # 例如 ['F0_base', 'F2_ltv']；None 表示全部变体
OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / EXPERIMENT_ID

FAST_PARAMS = dict(
    iterations=1600, depth=8, learning_rate=0.05, l2_leaf_reg=3,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    use_best_model=False, allow_writing_files=False, verbose=False,
)
VARIANTS = {
    'F0_base': [],
    'F1_dti': ['dti'],
    'F2_ltv': ['ltv'],
    'F3_income': ['income'],
    'F4_missing': ['missing'],
    'F5_freq': ['freq'],
    'F6_all': ['dti', 'ltv', 'income', 'missing', 'freq'],
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')

In [ ]:
if TRAIN['ablation_round1']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'feature block ablation, fast GPU CatBoost, paired vs F0_base',
        'seed': SEED,
        'fast_params': {k: v for k, v in FAST_PARAMS.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })

    selected = {n: b for n, b in VARIANTS.items() if RUN_ONLY is None or n in RUN_ONLY}
    for fold in FOLDS:
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        for name, blocks in selected.items():
            result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if result_path.is_file():
                saved = json.loads(result_path.read_text(encoding='utf-8'))
                require(saved['blocks'] == blocks, f'{name} 的定义已改变；请改变体名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            train, _, cats = variant_frames(blocks)
            started = time.perf_counter()
            model = catboost.CatBoostClassifier(**FAST_PARAMS)
            model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
            require(int(model.tree_count_) == FAST_PARAMS['iterations'], '轮数不符')
            probability = checked_probabilities(model, train.iloc[valid_idx])
            actual = y.iloc[valid_idx]
            pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                          'prob': probability}).to_csv(
                OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
            result = {
                'variant': name, 'blocks': blocks, 'fold': fold,
                'feature_count': train.shape[1],
                'valid_auc': roc_auc_score(actual, probability),
                'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
                'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                'finished_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            write_json(result_path, result)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
                  f'{train.shape[1]} 列  {result["seconds"]/60:.1f} 分钟', flush=True)
            del model, train
            gc.collect()

    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                            for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
    require(not results.empty, '还没有完成的结果')
    auc = results.pivot(index='fold', columns='variant', values='valid_auc')
    require('F0_base' in auc.columns, '缺少基线 F0_base')
    delta = auc.sub(auc['F0_base'], axis=0)
    table = pd.DataFrame({
        'folds_done': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'mean_delta_vs_F0': delta.mean(),
        'folds_better_than_F0': (delta > 0).sum(),
        'mean_logloss': results.groupby('variant')['valid_logloss'].mean(),
        'feature_count': results.groupby('variant')['feature_count'].first(),
    })
    table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_F0'] >= 0.0003)
                     & (table['folds_better_than_F0'] >= 4))
    table.loc['F0_base', 'keep'] = False
    table = table.sort_values('mean_delta_vs_F0', ascending=False)
    table.to_csv(OUTPUT_DIR / 'summary.csv')
    display(table.round(6))
    display(delta.round(6))
else:
    print("训练开关关闭（TRAIN['ablation_round1'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
runs = load_runs(ARTIFACTS / '02_feature_engineering' / 'ablation_round1')
table, delta = paired_table(runs, 'variant', 'F0_base', min_gain=0.0003)
table['feature_count'] = runs.groupby('variant')['feature_count'].first()
display(table.round(6))
plot_deltas(delta, 'F0_base', 'Feature ablation, round 1')

**结论**：全部特征块一起 **+0.0030（5/5 折）**；单独看，LTV 重算（`ltv_calc`、`ltv_gap`）**+0.0021（5/5）** 是主要来源，其余单独几乎无效，地区频次略差。

### 6.2 第二轮（基线 G0 = 第一轮全部特征）

保留标准：平均 ≥ +0.0003 且 ≥ 4/5 折。

In [ ]:
EXPERIMENT_ID = 'ablation_round2'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None             # 例如 ['F0_base', 'F2_ltv']；None 表示全部变体
OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / EXPERIMENT_ID

FAST_PARAMS = dict(
    iterations=1600, depth=8, learning_rate=0.05, l2_leaf_reg=3,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    use_best_model=False, allow_writing_files=False, verbose=False,
)
ROUND1 = ['dti', 'ltv', 'income', 'missing', 'freq']     # 第一轮全部特征（F6_all）
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
VARIANTS = {
    'G0_all': ROUND1,
    'G1_precision': ROUND1 + ['precision'],
    'G2_thresholds': ROUND1 + ['thresholds'],
    'G3_implied': ROUND1 + ['implied'],
    'G4_term': ROUND1 + ['term'],
    'G5_round2_all': ROUND1 + ['precision', 'thresholds', 'implied', 'term'],
    # 调参后去掉地区频次的五折与公开榜都略好：检验第二轮特征加在无频次基础上
    'G6_nofreq': NO_FREQ,
    'G7_nofreq_round2_all': NO_FREQ + ['precision', 'thresholds', 'implied', 'term'],
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')

In [ ]:
if TRAIN['ablation_round2']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'feature block ablation, fast GPU CatBoost, paired vs F0_base',
        'seed': SEED,
        'fast_params': {k: v for k, v in FAST_PARAMS.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })

    selected = {n: b for n, b in VARIANTS.items() if RUN_ONLY is None or n in RUN_ONLY}
    for fold in FOLDS:
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        for name, blocks in selected.items():
            result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if result_path.is_file():
                saved = json.loads(result_path.read_text(encoding='utf-8'))
                require(saved['blocks'] == blocks, f'{name} 的定义已改变；请改变体名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            train, _, cats = variant_frames(blocks)
            started = time.perf_counter()
            model = catboost.CatBoostClassifier(**FAST_PARAMS)
            model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
            require(int(model.tree_count_) == FAST_PARAMS['iterations'], '轮数不符')
            probability = checked_probabilities(model, train.iloc[valid_idx])
            actual = y.iloc[valid_idx]
            pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                          'prob': probability}).to_csv(
                OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
            result = {
                'variant': name, 'blocks': blocks, 'fold': fold,
                'feature_count': train.shape[1],
                'valid_auc': roc_auc_score(actual, probability),
                'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
                'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                'finished_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            write_json(result_path, result)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
                  f'{train.shape[1]} 列  {result["seconds"]/60:.1f} 分钟', flush=True)
            del model, train
            gc.collect()

    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                            for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
    require(not results.empty, '还没有完成的结果')
    auc = results.pivot(index='fold', columns='variant', values='valid_auc')
    require('G0_all' in auc.columns, '缺少基线 G0_all')
    delta = auc.sub(auc['G0_all'], axis=0)
    table = pd.DataFrame({
        'folds_done': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'mean_delta_vs_G0': delta.mean(),
        'folds_better_than_G0': (delta > 0).sum(),
        'mean_logloss': results.groupby('variant')['valid_logloss'].mean(),
        'feature_count': results.groupby('variant')['feature_count'].first(),
    })
    table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_G0'] >= 0.0003)
                     & (table['folds_better_than_G0'] >= 4))
    table.loc['G0_all', 'keep'] = False
    table = table.sort_values('mean_delta_vs_G0', ascending=False)
    table.to_csv(OUTPUT_DIR / 'summary.csv')
    display(table.round(6))
    display(delta.round(6))
else:
    print("训练开关关闭（TRAIN['ablation_round2'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
runs = load_runs(ARTIFACTS / '02_feature_engineering' / 'ablation_round2')
table, delta = paired_table(runs, 'variant', 'G0_all', min_gain=0.0003)
table['feature_count'] = runs.groupby('variant')['feature_count'].first()
display(table.round(6))
plot_deltas(delta, 'G0_all', 'Feature ablation, round 2')

**结论**：**G7（去掉地区频次 + 全部第二轮特征，72 列）+0.00125（5/5）** 最好；LTV 申报精度 +0.00075（4/5）、LTV 产品门槛 +0.00074（5/5）单独有效；反推房价、期限单独不确定。

### 6.3 第三轮（基线 H0 = G7）

不同 Colab 会话之间同一配置可相差约 0.0005，所以本轮保留标准提高为：平均 ≥ +0.0005 且 ≥ 4/5 折。

In [ ]:
EXPERIMENT_ID = 'ablation_round3'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None             # 例如 ['F0_base', 'F2_ltv']；None 表示全部变体
OUTPUT_DIR = ARTIFACTS / '02_feature_engineering' / EXPERIMENT_ID

FAST_PARAMS = dict(
    iterations=1600, depth=8, learning_rate=0.05, l2_leaf_reg=3,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    use_best_model=False, allow_writing_files=False, verbose=False,
)
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
G7 = NO_FREQ + ['precision', 'thresholds', 'implied', 'term']   # 第二轮的最佳特征集（72 列）
ROUND3 = ['purpose', 'ltv_consistency', 'risk_layers', 'tract_income']
VARIANTS = {
    'H0_G7': G7,
    'H1_purpose': G7 + ['purpose'],
    'H2_ltv_consistency': G7 + ['ltv_consistency'],
    'H3_risk_layers': G7 + ['risk_layers'],
    'H4_tract_income': G7 + ['tract_income'],
    'H5_round3_all': G7 + ROUND3,
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')

In [ ]:
if TRAIN['ablation_round3']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'feature block ablation, fast GPU CatBoost, paired vs F0_base',
        'seed': SEED,
        'fast_params': {k: v for k, v in FAST_PARAMS.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })

    selected = {n: b for n, b in VARIANTS.items() if RUN_ONLY is None or n in RUN_ONLY}
    for fold in FOLDS:
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        for name, blocks in selected.items():
            result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if result_path.is_file():
                saved = json.loads(result_path.read_text(encoding='utf-8'))
                require(saved['blocks'] == blocks, f'{name} 的定义已改变；请改变体名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            train, _, cats = variant_frames(blocks)
            started = time.perf_counter()
            model = catboost.CatBoostClassifier(**FAST_PARAMS)
            model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
            require(int(model.tree_count_) == FAST_PARAMS['iterations'], '轮数不符')
            probability = checked_probabilities(model, train.iloc[valid_idx])
            actual = y.iloc[valid_idx]
            pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                          'prob': probability}).to_csv(
                OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
            result = {
                'variant': name, 'blocks': blocks, 'fold': fold,
                'feature_count': train.shape[1],
                'valid_auc': roc_auc_score(actual, probability),
                'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
                'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                'finished_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            write_json(result_path, result)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
                  f'{train.shape[1]} 列  {result["seconds"]/60:.1f} 分钟', flush=True)
            del model, train
            gc.collect()

    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                            for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
    require(not results.empty, '还没有完成的结果')
    auc = results.pivot(index='fold', columns='variant', values='valid_auc')
    require('H0_G7' in auc.columns, '缺少基线 H0_G7')
    delta = auc.sub(auc['H0_G7'], axis=0)
    table = pd.DataFrame({
        'folds_done': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'mean_delta_vs_H0': delta.mean(),
        'folds_better_than_H0': (delta > 0).sum(),
        'mean_logloss': results.groupby('variant')['valid_logloss'].mean(),
        'feature_count': results.groupby('variant')['feature_count'].first(),
    })
    table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_H0'] >= 0.0005)
                     & (table['folds_better_than_H0'] >= 4))
    table.loc['H0_G7', 'keep'] = False
    table = table.sort_values('mean_delta_vs_H0', ascending=False)
    table.to_csv(OUTPUT_DIR / 'summary.csv')
    display(table.round(6))
    display(delta.round(6))
else:
    print("训练开关关闭（TRAIN['ablation_round3'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
runs = load_runs(ARTIFACTS / '02_feature_engineering' / 'ablation_round3')
table, delta = paired_table(runs, 'variant', 'H0_G7', min_gain=0.0005)
table['feature_count'] = runs.groupby('variant')['feature_count'].first()
display(table.round(6))
plot_deltas(delta, 'H0_G7', 'Feature ablation, round 3')

**结论**：第三轮最好只有 +0.00035（3/5），均未达标，与残差诊断（05）一致——特征工程到此饱和。

## 7. 最终使用的特征集

| 特征集 | 组成 | 列数 | 用于 |
|---|---|---:|---|
| **G7** | 原始 56 列 + `dti`、`ltv`、`income`、`missing`、`precision`、`thresholds`、`implied`、`term` | 72 | 最终模型 M1、M2 |
| **LEAN** | 原始 56 列 + `dti`、`ltv`、`income`、`missing`、`precision`、`thresholds` | 68 | 最终模型 M3 |
| ROUND1_ALL | 原始 56 列 + 第一轮全部 | 68 | 对照 M0 |

第三轮特征保存在特征库中，但没有进入最终模型。